# Programación orientada a objetos con Python
## Video 4. Encapsulamiento: convención de privacidad, `@property` e invariantes

**Hilo conductor de la serie:** construiremos objetos matemáticos (fracciones, vectores y polinomios) y los iremos ampliando de video en video.

**Al terminar este video podrás:**
1. Definir formalmente qué es un invariante de clase y cómo puede romperse si los atributos quedan expuestos a modificaciones directas.
2. Aplicar la convención de privacidad en Python (`_atributo` y `__atributo`) para proteger la representación interna.
3. Exponer atributos de solo lectura y atributos con validación mediante `@property` y `@<nombre>.setter`.
4. Diseñar propiedades calculadas bajo demanda (como la norma de un vector o el grado de un polinomio) sin duplicar ni desincronizar el estado.

## 1. Planteamiento del problema

**Definición.** Un *invariante de clase* es una condición lógica sobre el estado de una instancia que se cumple al terminar `__init__` y se conserva verdadera antes y después de cualquier operación pública sobre el objeto.

En nuestros objetos matemáticos los invariantes tienen un significado preciso:
- En `Fraccion`, representando un par $(p, q) \in \mathbb{Z} \times \mathbb{Z}$, el invariante exige $q > 0$ y $\gcd(p, q) = 1$.
- En un intervalo cerrado $[a, b] \subset \mathbb{R}$, el invariante exige $a \le b$.
- En `Polinomio`, el invariante exige que la secuencia de coeficientes $[a_0, a_1, \dots, a_n]$ no esté vacía y no conserve ceros finales redundantes, salvo el polinomio nulo $[0]$.

**Problema: protección de invariantes mediante encapsulamiento.**

- **Entrada:** instancias válidas de `Fraccion`, `VectorPlano`, `Polinomio` e `IntervaloCerrado`.
- **Salida:** clases encapsuladas cuya interfaz pública impida corromper el estado interno mediante asignaciones directas o mutaciones accidentales.

## 2. Violación del invariante cuando el estado es público

Cuando los atributos se exponen directamente sin control, cualquier código externo puede modificarlos después de que `__init__` haya terminado. Aunque `__init__` haya validado las entradas al construir el objeto, el invariante queda desprotegido durante el resto de la vida de la instancia.

In [ ]:
from math import gcd, sqrt


class FraccionAbierta:
    def __init__(self, numerador, denominador):
        if denominador == 0:
            raise ValueError("El denominador no puede ser cero.")
        if denominador < 0:
            numerador, denominador = -numerador, -denominador
        divisor = gcd(numerador, denominador)
        self.numerador = numerador // divisor
        self.denominador = denominador // divisor

    def mostrar(self):
        return f"{self.numerador}/{self.denominador}"


class PolinomioAbierto:
    def __init__(self, coeficientes):
        lista = list(coeficientes)
        if not lista:
            raise ValueError("La lista de coeficientes no puede estar vacía.")
        while len(lista) > 1 and lista[-1] == 0:
            lista.pop()
        self.coeficientes = lista

    def grado(self):
        return len(self.coeficientes) - 1


f = FraccionAbierta(1, 2)
f.denominador = 0  # rompe el invariante: denominador distinto de cero
print("Fracción corrupta:", f.mostrar())

g = FraccionAbierta(1, 2)
g.numerador = 2    # rompe la forma irreducible: ahora vale 2/2
print("Fracción sin simplificar:", g.mostrar())

p = PolinomioAbierto([1, 3])
print("Grado inicial de p:", p.grado())
p.coeficientes.extend([0, 0])  # mutación externa de la lista interna
print("Grado aparente después de mutar la lista:", p.grado())

## 3. Convención de privacidad (`_atributo`) y transformación de nombres (`__atributo`)

En Python no existen modificadores de acceso estrictos como `private` en otros lenguajes; el lenguaje confía en dos mecanismos de señalización y aislamiento:

1. **Un guion bajo inicial (`self._atributo`):** es la convención universal entre programadores de Python para indicar que el atributo forma parte de la representación interna de la clase y no debe modificarse directamente desde fuera.
2. **Doble guion bajo inicial (`self.__atributo`):** activa la *transformación de nombres* (*name mangling*): Python renombra internamente el atributo a `_NombreClase__atributo` para evitar colisiones accidentales en subclases.

In [ ]:
class RegistroInterno:
    def __init__(self, valor):
        self._valor_interno = valor
        self.__sello_privado = "invariante-verificado"

    def consultar_sello(self):
        return self.__sello_privado


registro = RegistroInterno(42)
print("Diccionario de atributos de la instancia:", registro.__dict__)
print("Acceso controlado mediante método:", registro.consultar_sello())

## 4. Propiedades (`@property`), validadores (`@setter`) y atributos calculados

El decorador `@property` permite exponer un método como si fuera un atributo de lectura (`objeto.atributo`), conservando el control total dentro de la clase:

- **Propiedad de solo lectura:** se define únicamente el método decorado con `@property`. Intentar asignar `objeto.atributo = valor` lanza automáticamente `AttributeError`. Es el diseño ideal para objetos matemáticos inmutables como `Fraccion` y `Polinomio`.
- **Propiedad con validación (`@nombre.setter`):** si además se define un método decorado con `@nombre.setter`, toda asignación `objeto.nombre = valor` pasa por esa función de validación antes de actualizar el atributo interno.
- **Propiedad calculada:** un valor derivado del estado (como la norma de un vector o el grado de un polinomio) se calcula a partir de los atributos internos fundamentales, evitando almacenar copias redundantes que puedan quedar desincronizadas.

In [ ]:
class Fraccion:
    def __init__(self, numerador, denominador):
        if not isinstance(numerador, int) or not isinstance(denominador, int):
            raise TypeError("Numerador y denominador deben ser números enteros.")
        if denominador == 0:
            raise ValueError("El denominador no puede ser cero.")
        if denominador < 0:
            numerador, denominador = -numerador, -denominador
        divisor = gcd(numerador, denominador)
        self._numerador = numerador // divisor
        self._denominador = denominador // divisor

    @property
    def numerador(self):
        return self._numerador

    @property
    def denominador(self):
        return self._denominador

    @property
    def valor_flotante(self):
        return self._numerador / self._denominador

    def sumar(self, otra):
        return Fraccion(
            self._numerador * otra._denominador + self._denominador * otra._numerador,
            self._denominador * otra._denominador,
        )

    def mostrar(self):
        return f"{self._numerador}/{self._denominador}"


f = Fraccion(6, -8)
print("Fracción:", f.mostrar(), "| numerador:", f.numerador, "| denominador:", f.denominador)
print("Valor decimal aproximado:", f.valor_flotante)

try:
    f.denominador = 0
except AttributeError as error:
    print("Protección de solo lectura activada:", error)

In [ ]:
class VectorPlano:
    dimension = 2

    def __init__(self, x, y):
        self.x = x
        self.y = y

    @property
    def x(self):
        return self._x

    @x.setter
    def x(self, valor):
        self._x = float(valor)

    @property
    def y(self):
        return self._y

    @y.setter
    def y(self, valor):
        self._y = float(valor)

    @property
    def norma(self):
        return sqrt(self._x ** 2 + self._y ** 2)

    @norma.setter
    def norma(self, nueva_norma):
        nueva_norma = float(nueva_norma)
        if nueva_norma < 0:
            raise ValueError("La norma de un vector no puede ser negativa.")
        norma_actual = self.norma
        if norma_actual == 0:
            if nueva_norma > 0:
                raise ValueError("No se puede reescalar el vector nulo a una norma positiva sin dirección.")
            return
        factor = nueva_norma / norma_actual
        self._x *= factor
        self._y *= factor

    def mostrar(self):
        return f"({self._x}, {self._y})"


class Polinomio:
    variable = "x"

    def __init__(self, coeficientes):
        lista = list(coeficientes)
        if not lista:
            raise ValueError("La secuencia de coeficientes no puede estar vacía.")
        while len(lista) > 1 and lista[-1] == 0:
            lista.pop()
        self._coeficientes = tuple(lista)

    @property
    def coeficientes(self):
        return self._coeficientes

    @property
    def grado(self):
        return len(self._coeficientes) - 1


v = VectorPlano(3, 4)
print("Vector inicial:", v.mostrar(), "| norma:", v.norma)
v.norma = 10.0
print("Vector reescalado a norma 10:", v.mostrar(), "| nueva norma:", round(v.norma, 6))

try:
    v.norma = -2.0
except ValueError as error:
    print("Validación de norma negativa:", error)

pol = Polinomio([1, 0, 2, 0, 0])
print("Coeficientes inmutables:", pol.coeficientes, "| grado:", pol.grado)

### Tabla comparativa

| Mecanismo | Lectura externa | Escritura externa | Protección del invariante | Caso de uso matemático |
|---|---|---|---|---|
| Atributo público directo (`self.x`) | Directa | Libre, sin validación | Ninguna | Estructuras simples sin restricciones |
| Propiedad de solo lectura (`@property`) | Como atributo | Prohibida (`AttributeError`) | Total (inmutabilidad) | `Fraccion.numerador`, `Polinomio.coeficientes` |
| Propiedad con validación (`@property` + `@setter`) | Como atributo | Controlada por el validador | Total mediante excepciones | Coordenadas reales, extremos $[a, b]$ con $a \le b$ |
| Propiedad calculada (`@property` derivada) | Como atributo | Opcional o prohibida | Evita datos duplicados desincronizados | `VectorPlano.norma`, `Polinomio.grado` |

## 5. Ejercicios

**Ejercicio 1 (`IntervaloCerrado` con invariante $a \le b$).** Diseña una clase `IntervaloCerrado` que represente un intervalo cerrado $[a, b] \subset \mathbb{R}$:
- **Entrada:** dos números reales `izquierdo` ($a$) y `derecho` ($b$) en `__init__`, con la restricción $a \le b$.
- **Salida:** instancias con atributos internos `_izquierdo` y `_derecho`, propiedades `izquierdo` y `derecho` equipadas con `@setter` que rechacen con `ValueError` cualquier modificación que viole $a \le b$, y propiedades calculadas de solo lectura `longitud` ($b - a$) y `punto_medio` ($\frac{a + b}{2}$). Incluye también el método `contiene(self, punto)` que devuelva `True` si $a \le \text{punto} \le b$.

**Ejercicio 2 (`Fraccion` encapsulada con `es_entera` y `reciproco`).** Amplía la clase `Fraccion` encapsulada del texto añadiendo:
- Una propiedad calculada `es_entera` que devuelva `True` cuando el denominador normalizado sea igual a `1`.
- El método `reciproco(self)` que devuelva la fracción inversa $(q, p)$ o lance `ZeroDivisionError` si el numerador es `0`.

**Ejercicio 3 (`Polinomio` encapsulado con `coeficiente_principal` y `es_monico`).** Amplía la clase `Polinomio` encapsulada añadiendo:
- Una propiedad calculada `coeficiente_principal` que devuelva el coeficiente de mayor grado $a_n$ (es decir, `self._coeficientes[-1]`).
- Una propiedad calculada `es_monico` que devuelva `True` si y solo si `self.coeficiente_principal == 1`.

## 6. Soluciones

In [ ]:
# Ejercicio 1
class IntervaloCerrado:
    def __init__(self, izquierdo, derecho):
        izquierdo = float(izquierdo)
        derecho = float(derecho)
        if izquierdo > derecho:
            raise ValueError("El extremo izquierdo no puede superar al extremo derecho.")
        self._izquierdo = izquierdo
        self._derecho = derecho

    @property
    def izquierdo(self):
        return self._izquierdo

    @izquierdo.setter
    def izquierdo(self, valor):
        valor = float(valor)
        if valor > self._derecho:
            raise ValueError("El extremo izquierdo no puede superar al extremo derecho.")
        self._izquierdo = valor

    @property
    def derecho(self):
        return self._derecho

    @derecho.setter
    def derecho(self, valor):
        valor = float(valor)
        if valor < self._izquierdo:
            raise ValueError("El extremo derecho no puede ser menor que el izquierdo.")
        self._derecho = valor

    @property
    def longitud(self):
        return self._derecho - self._izquierdo

    @property
    def punto_medio(self):
        return (self._izquierdo + self._derecho) / 2.0

    def contiene(self, punto):
        return self._izquierdo <= punto <= self._derecho

    def mostrar(self):
        return f"[{self._izquierdo}, {self._derecho}]"


intervalo = IntervaloCerrado(2, 6)
print("Intervalo:", intervalo.mostrar(), "| longitud:", intervalo.longitud, "| punto medio:", intervalo.punto_medio)
intervalo.izquierdo = 4
print("Tras mover izquierdo a 4:", intervalo.mostrar(), "| longitud:", intervalo.longitud)
print("¿Contiene a 5?", intervalo.contiene(5))

try:
    intervalo.derecho = 1
except ValueError as error:
    print("Invariante protegido:", error)

In [ ]:
# Ejercicio 2
class Fraccion:
    def __init__(self, numerador, denominador):
        if not isinstance(numerador, int) or not isinstance(denominador, int):
            raise TypeError("Numerador y denominador deben ser números enteros.")
        if denominador == 0:
            raise ValueError("El denominador no puede ser cero.")
        if denominador < 0:
            numerador, denominador = -numerador, -denominador
        divisor = gcd(numerador, denominador)
        self._numerador = numerador // divisor
        self._denominador = denominador // divisor

    @property
    def numerador(self):
        return self._numerador

    @property
    def denominador(self):
        return self._denominador

    @property
    def es_entera(self):
        return self._denominador == 1

    def reciproco(self):
        if self._numerador == 0:
            raise ZeroDivisionError("El cero no tiene recíproco.")
        return Fraccion(self._denominador, self._numerador)

    def mostrar(self):
        return f"{self._numerador}/{self._denominador}"


a = Fraccion(10, 2)
b = Fraccion(3, 5)
print("10/2 es:", a.mostrar(), "| ¿es entera?", a.es_entera)
print("3/5 es:", b.mostrar(), "| ¿es entera?", b.es_entera, "| recíproco:", b.reciproco().mostrar())

In [ ]:
# Ejercicio 3
class Polinomio:
    variable = "x"

    def __init__(self, coeficientes):
        lista = list(coeficientes)
        if not lista:
            raise ValueError("La secuencia de coeficientes no puede estar vacía.")
        while len(lista) > 1 and lista[-1] == 0:
            lista.pop()
        self._coeficientes = tuple(lista)

    @property
    def coeficientes(self):
        return self._coeficientes

    @property
    def grado(self):
        return len(self._coeficientes) - 1

    @property
    def coeficiente_principal(self):
        return self._coeficientes[-1]

    @property
    def es_monico(self):
        return self.coeficiente_principal == 1


p1 = Polinomio([-6, 11, -6, 1, 0])
p2 = Polinomio([2, 3, 5])
print("p1:", p1.coeficientes, "| grado:", p1.grado, "| principal:", p1.coeficiente_principal, "| ¿es mónico?", p1.es_monico)
print("p2:", p2.coeficientes, "| grado:", p2.grado, "| principal:", p2.coeficiente_principal, "| ¿es mónico?", p2.es_monico)

## 7. Resumen

- Un **invariante de clase** es una propiedad matemática que toda instancia válida debe respetar durante toda su existencia.
- Los atributos internos precedidos por `_` y las estructuras inmutables (como `tuple`) evitan mutaciones accidentales desde el exterior.
- El decorador `@property` convierte métodos en atributos de solo lectura o calculados bajo demanda, mientras que `@<nombre>.setter` centraliza las validaciones al asignar nuevos valores.
- En el siguiente video integraremos nuestros objetos con la sintaxis propia de Python mediante los métodos especiales (`__repr__`, `__str__`, `__eq__`, `__lt__`, `__add__` y `__len__`).